Download each CSV file for each month

In [33]:
import requests

for year in range(2026, 2027):
    for month in range(1,13):
        if year == 2026 and month > 7:
            break
        print(f"Downloading data for {year}-{month:02d}...")
        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        open(f"ontime_{year}_{month}.zip", "wb").write(r.content)

Import some libraries, including snowflake connector

In [ ]:
import io, gzip, zipfile, snowflake.connector

Create the connection to snowflake and put the credetials in a separate file.
Also created a warehouse in snowflake, a database and a schema to store the data.

In [36]:
import credentials
conn = snowflake.connector.connect(
    user=credentials.credentials['user'],
    password=credentials.credentials['password'],
    account=credentials.credentials['account'],
    warehouse=credentials.credentials['warehouse'],
    database=credentials.credentials['database'],
    schema=credentials.credentials['schema']
)

Before the next step I also created a STAGE inside snowflake to store the stage data or raw data.

Unzip the downloaded files locally and push them to the snowflake staging area called @staging_data. 

In [22]:
import os, re, io, gzip, zipfile
for file in os.listdir():
    print(f"Found {file}")
    if file.startswith("ontime_") and file.endswith(".zip"):
        print(f"Processing {file} ...")
        with open(file, "rb") as f:
            zip_data = f.read()
            with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
                for name in z.namelist():
                    if name.endswith(".csv"):
                        # clean output name: no parentheses, ends in .csv 
                        out = re.sub(r"[()]", "", os.path.basename(name)) + ".gz"
                        with z.open(name) as csv_file, gzip.open(out, "wt", encoding="utf-8", newline="") as dst:
                            for line in io.TextIOWrapper(csv_file, encoding="utf-8", errors="replace", newline=""):
                                line = line.rstrip("\r\n")
                                if line.endswith(","):
                                    line = line[:-1]          # drop only the one trailing comma
                                dst.write(line + "\n")
                        print(f"Uploading {out}")
                        conn.cursor().execute(f"PUT 'file://{out}' @staging_data AUTO_COMPRESS=FALSE OVERWRITE=TRUE")

So after we downloaded all the files there is an extra step if we are using a trial account. We ideally want to upload this files to snowflake. If not using a trial account, we can directly run this notebook inside snowflake, perhaps it needs some folder creation and changes to code.

After we successfully moved the files to snowflake, now we need to remove them locally.

In [ ]:
for file in os.listdir():
    if file.endswith("gz") or file.endswith("zip"):
        print(f"Removing {file} ...")
        os.remove(file)

Next step is to create the weather tables for precipitation, temperatures, and wind
But first I am going to clone the dbt-snowflake getting started project to this repo and link github with snowflake account.

In [34]:
import requests

url = 'https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_haVdhR_PNeeVRef'
r = requests.get(url, timeout=120)
r.raise_for_status()
open("carrier_lookup.csv", "wb").write(r.content)

54240

In [37]:
file_name = "carrier_lookup.csv"
print(f"Uploading {file_name}")
conn.cursor().execute(f"PUT 'file://{file_name}' @staging_data AUTO_COMPRESS=FALSE OVERWRITE=TRUE")

Uploading carrier_lookup.csv


In [43]:
airport_lookup_url = "https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FLL&QO_fu146_anzr=N8vn6v10+f722146+gnoyr5"
r = requests.get(airport_lookup_url, timeout=120)
r.raise_for_status()
open("airport_lookup.zip", "wb").write(r.content)

131280

After running the download_t_master_cord_selenium.py we downloaded the lookup data for airport to coordinates. So we still need to unzip it and send to snowflake

In [58]:
zip_path = Path("data/bts/_downloads/T_MASTER_CORD_20261021_155126.zip")
out_dir = Path("data/bts")

with zipfile.ZipFile(zip_path) as z:
    print("Files in zip:", z.namelist())
    csv_name = next(n for n in z.namelist() if n.upper().endswith("T_MASTER_CORD.CSV"))
    csv_path = Path(z.extract(csv_name, out_dir))

print(f"Extracted: {csv_path}")

Files in zip: ['Term.csv', 'T_MASTER_CORD.csv']
Extracted: data/bts/T_MASTER_CORD.csv


Now Lets send this file to staging_data in snowflake

In [ ]:
conn.cursor().execute(f"PUT 'file://out_dir/T_MASTER_CORD.CSV' @staging_data AUTO_COMPRESS=FALSE OVERWRITE=TRUE")